# NeuroTrace V6 GPU workflow — A100/T4/P100

**Research pipeline, not a validated diagnostic.** Requires authorized continuous
trajectory data, reviewed labels and a locked subject partition. Synthetic tremor
is a pseudo-label ablation, not evidence of mobile-PD sensitivity. Default generates
5001 touch +5001 mouse variants (10002 variants, not people). Both donor and recipient
must remain in each training fold. Missing real data/credentials cause a visible
blocker; this notebook never substitutes mock fixtures.

Keep PR #4 open; nothing here pushes, merges or overwrites V5. Enable GPU (Colab
Runtime → Change runtime type; Kaggle accelerator P100 and Internet). P100 fp16 is
supported by pinned PyTorch2.5.1; driver compatibility must still be checked.

In [4]:
from pathlib import Path
import os, subprocess, sys
REPO_DIR = Path('/kaggle/working/NeuroTrace-AI' if Path('/kaggle').exists() else '/content/NeuroTrace-AI')
if not (REPO_DIR / '.git').exists():
    subprocess.check_call(['git','clone','--branch','arena/4edc99ce-neurotrace-ai',
        'https://github.com/VaKhang0709/NeuroTrace-AI.git', str(REPO_DIR)])
os.chdir(REPO_DIR)
branch = subprocess.check_output(['git','branch','--show-current'], text=True).strip()
assert branch == 'arena/4edc99ce-neurotrace-ai', f'Wrong branch: {branch}; do not train here'
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

Commit: 2d94669c55893a333a18b7dd1a831ede660e373d


## Install before importing torch

Use a fresh runtime. If torch was already imported, restart after installation.
Downloads may be several GB. No credentials belong in notebook cells or outputs.

In [5]:
subprocess.check_call([sys.executable,'-m','pip','install','torch==2.5.1'])
subprocess.check_call([sys.executable,'-m','pip','install','-r','requirements-v6.txt'])
import torch
from scripts.train_v6_gpu_standalone import gpu_info
gpu_info(require=True)
# Microbatch2 is a conservative starting point, not measured VRAM guidance.
print('cuDNN:',torch.backends.cudnn.version())

{
  "name": "Tesla T4",
  "vram_gib": 14.56,
  "capability": [
    7,
    5
  ],
  "torch": "2.5.1+cu124"
}
cuDNN: 90100


## Authorized private inputs

Colab: mount Drive using the optional cell below, or upload a reviewed dataset.
Kaggle: attach a **private** input dataset. `RECORDS` is a versioned unified JSON
list containing real HC touch/mouse and/or UCI records. If UCI is absent from that
list, downloader + strict raw parser supplies it. `PARTITION` contains disjoint
`development_subjects`, `holdout_subjects`, optional `unlabeled_subjects`. Human
identity linkage across devices is the operator's responsibility. No test subjects
may enter SSL or synthetic donor/recipient pools. Never create a random drawing split.

In [6]:
# OPTIONAL Colab-only authentication, performed by the operator (not embedded secrets):
# from google.colab import drive
# drive.mount('/content/drive')
PRIVATE_ROOT = Path('/kaggle/input/neurotrace-v6-private' if Path('/kaggle').exists()
                    else '/content/drive/MyDrive/NEUROTRACE/v6')
RECORDS = PRIVATE_ROOT / 'unified_records.json'
PARTITION = PRIVATE_ROOT / 'partition.json'
LOCAL_UCI = None  # Optional authorized .zip or extracted directory in private storage.
MIRRORS = None    # Optional reviewed DOI/direct-HTTPS/checksum/license JSON manifest.
PER_DOMAIN = 5001
EPOCHS, SSL_EPOCHS, BATCH_SIZE = 200, 100, 2
USE_SYNTHETIC = False # Baseline first; True enables explicitly experimental pseudo-PD ablation.
FEATURE_CONTRACT = 'v6-dwt19-v1' # New requested detail-level channels, NOT bands19 checkpoints.
assert RECORDS.is_file() and PARTITION.is_file(), 'Missing real authorized records/partition; no mock fallback'
OUTPUT = Path('models/v6/checkpoints')
PREPARED = Path('data/processed/v6_unified_records.json')

AssertionError: Missing real authorized records/partition; no mock fallback

## Acquire and generate (CPU preparation)

UCI official download is attempted. Network denial is reported; provide LOCAL_UCI
or include verified UCI records. Zenodo/Figshare/PhysioNet fetchers require reviewed
open dynamic-trajectory metadata, never image-to-kinematics conversion. Missing
recorded UCI-PD or HC touch/mouse blocks synthesis. Existing immutable manifests
are reused only after provenance/count checks. Keep all generated data private.

In [ ]:
common = [sys.executable,'-m','scripts.train_v6_gpu_standalone',
          '--records',str(RECORDS),'--partition',str(PARTITION),
          '--prepared-records',str(PREPARED),'--output',str(OUTPUT),
          '--per-domain',str(PER_DOMAIN),'--feature-contract',FEATURE_CONTRACT,
          '--epochs',str(EPOCHS),'--ssl-epochs',str(SSL_EPOCHS),'--batch-size',str(BATCH_SIZE)]
if LOCAL_UCI: common += ['--local-uci',str(LOCAL_UCI)]
if MIRRORS: common += ['--mirrors',str(MIRRORS)]
subprocess.check_call(common + ['--prepare-only'])

## Five-fold GPU training

Default full model is15.78M parameters. CUDA AMP + accumulation target256; five-epoch
warmup/cosine restarts. SSL is fold-scoped. Validation is real-only, subject-primary.
Synthetic ablation filters **both parents** to training and samples at most25% of
training windows by default. Generation count does not imply every variant appears
in every epoch. Validation CSV records ROC-AUC,PR-AUC,sensitivity,specificity,ECE.
Model/optimizer checkpoints may consume several GB. Preserve outputs privately.

In [ ]:
train_command = common + (['--use-synthetic','--synthetic-fraction','0.25'] if USE_SYNTHETIC else [])
subprocess.check_call(train_command)
for fold in range(5):
    assert (OUTPUT / f'fold_{fold}_best.pth').is_file()
print('Training completed; inspect all fold CSVs. No clinical claim is implied.')

## Member ONNX export + validation-only calibration

Selects best member by validation AUC, not holdout. Exports dynamic batch/time,
checks100tensors across lengths256/384/512/768. Real training remains512steps.
Isotonic calibration is AFTER subject-mean probabilities; validation-selected Youden
threshold is a V6-only research operating point, not a clinical optimum. Use
`fixed_0.5` for a prespecified fixed threshold. Scaler JSON clearly distinguishes
measured training statistics from per-window normalization: **do not apply global
mean/std a second time**. No changes to V5 root artifacts.

In [ ]:
checkpoints = [str(OUTPUT / f'fold_{fold}_best.pth') for fold in range(5)]
subprocess.check_call([sys.executable,'-m','scripts.export_v6_onnx_pipeline',
    '--checkpoints',*checkpoints,'--records',str(PREPARED),'--partition',str(PARTITION),
    '--output','models/v6','--threshold-policy','fixed_0.5'])

## Locked holdout evaluation (no threshold search)

The existing evaluation protocol reports raw ensemble scores at0.50. Member isotonic
calibration/selected threshold is NOT silently applied to the ensemble. Ensemble
calibration requires a separate all-member-unseen cohort. Keep private predictions
and embeddings out of Git. Missing mobile PD means mobile sensitivity is unmeasurable.

In [ ]:
subprocess.check_call([sys.executable,'-m','src.inference.v6_ensemble_inference',
    '--records',str(PREPARED),'--partition',str(PARTITION),'--checkpoints',*checkpoints,
    '--output','reports/private/v6_predictions.npz'])
subprocess.check_call([sys.executable,'-m','src.evaluation.v6_eval',
    '--predictions','reports/private/v6_predictions.npz','--partition',str(PARTITION)])

## Preserve outputs / review

Copy `models/v6/`, private fold manifests and reports to your authorized Drive or
Kaggle private output storage. Do NOT upload raw trajectories, credentials, patient
IDs or unreviewed clinical predictions to public repositories. This notebook has no
`git push` or merge step. Tech Lead reviews PR #4 manually. Review
`docs/v6_gpu_execution_guide.md` and `BLOCKERS.md` before any model promotion.